# Readmission Baseline Model

Same process as the no-show baseline model: logistic regression baseline → class balancing → threshold tuning → gradient boosting comparison → feature importance.

## Load cleaned data

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/readmission_clean.csv')
df.shape

(99340, 50)

## Build features (X) and target (y)

Columns excluded from X:
- `encounter_id`, `patient_nbr` — identifiers, not predictive
- `readmitted` — the *original* 3-value target column. This isn't just correlated with the answer, it literally **is** the answer (`readmit_30_flag` was derived directly from it) — including it would be the most extreme possible leakage.
- `glu_tested`, `a1c_tested` — these were an earlier approach (binary tested-flags) that we superseded when we switched max_glu_serum/A1Cresult to a fuller categorical with a 'Not_Tested' level. Keeping both would be redundant.
- `readmit_30_flag` — this is our target, goes into y

In [2]:
X = df.drop(columns=['encounter_id', 'patient_nbr', 'readmitted', 'glu_tested', 'a1c_tested', 'readmit_30_flag'])
y = df['readmit_30_flag']
X.dtypes

race                          str
gender                        str
age                           str
admission_type_id             str
discharge_disposition_id      str
admission_source_id           str
time_in_hospital            int64
payer_code                    str
medical_specialty             str
num_lab_procedures          int64
num_procedures              int64
num_medications             int64
number_outpatient           int64
number_emergency            int64
number_inpatient            int64
number_diagnoses            int64
max_glu_serum                 str
A1Cresult                     str
metformin                     str
repaglinide                   str
nateglinide                   str
chlorpropamide                str
glimepiride                   str
acetohexamide                 str
glipizide                     str
glyburide                     str
tolbutamide                   str
pioglitazone                  str
rosiglitazone                 str
acarbose      

## Check categorical cardinality before encoding

Same check we did for Neighbourhood in the no-show dataset — want to catch any high-cardinality columns before one-hot encoding blows up the feature count.

In [3]:
cat_cols = X.select_dtypes(include='object').columns
X[cat_cols].nunique().sort_values(ascending=False)

/var/folders/71/ntvd5lc57cg2wzsrjb6lsb1r0000gn/T/ipykernel_22784/3750970879.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns


medical_specialty           73
discharge_disposition_id    20
payer_code                  18
admission_source_id         15
diag_3_cat                  10
diag_1_cat                  10
age                         10
diag_2_cat                  10
race                         6
admission_type_id            6
A1Cresult                    4
pioglitazone                 4
max_glu_serum                4
glyburide-metformin          4
insulin                      4
metformin                    4
acarbose                     4
rosiglitazone                4
miglitol                     4
glipizide                    4
repaglinide                  4
nateglinide                  4
glimepiride                  4
chlorpropamide               4
glyburide                    4
tolazamide                   3
gender                       2
metformin-pioglitazone       2
diabetesMed                  2
change                       2
glipizide-metformin          2
metformin-rosiglitazone      2
glimepir

In [4]:
X['medical_specialty'].value_counts().head(20)

medical_specialty
Unknown                              48614
InternalMedicine                     14237
Emergency/Trauma                      7419
Family/GeneralPractice                7252
Cardiology                            5278
Surgery-General                       3059
Nephrology                            1539
Orthopedics                           1392
Orthopedics-Reconstructive            1230
Radiologist                           1121
Pulmonology                            854
Psychiatry                             853
Urology                                682
ObstetricsandGynecology                669
Surgery-Cardiovascular/Thoracic        642
Gastroenterology                       538
Surgery-Vascular                       525
Surgery-Neuro                          462
PhysicalMedicineandRehabilitation      391
Oncology                               319
Name: count, dtype: int64

## Group rare medical_specialty categories into 'Other'

In [5]:
spec_counts = X['medical_specialty'].value_counts()
rare_specs = spec_counts[spec_counts < 500].index
X['medical_specialty'] = X['medical_specialty'].apply(lambda x: 'Other' if x in rare_specs else x)
X['medical_specialty'].nunique()

18

## One-hot encode and train/test split

In [6]:
X_encoded = pd.get_dummies(X, drop_first=True)
X_encoded.shape

(99340, 178)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, stratify=y, random_state=42
)
X_train.shape, X_test.shape

((79472, 178), (19868, 178))

## Baseline logistic regression (default, unbalanced)

Same starting point as the no-show model — establish the naive baseline before addressing class imbalance (recall this dataset's imbalance is worse: ~89/11 vs no-show's ~80/20).

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     17605
           1       0.49      0.02      0.04      2263

    accuracy                           0.89     19868
   macro avg       0.69      0.51      0.49     19868
weighted avg       0.84      0.89      0.84     19868

ROC-AUC: 0.6668822366602103


/Users/priyae/BA Projects/accessiq-patient-risk-dashboard/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Scale features for logistic regression

Fit the scaler on training data only, then apply (not re-fit) to test data, to avoid leaking test-set statistics into the scaling parameters.

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Retrain baseline logistic regression on scaled data

In [10]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     17605
           1       0.51      0.02      0.04      2263

    accuracy                           0.89     19868
   macro avg       0.70      0.51      0.49     19868
weighted avg       0.84      0.89      0.84     19868

ROC-AUC: 0.6674694086600904


## Logistic regression with class_weight='balanced'

In [11]:
model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
model_balanced.fit(X_train_scaled, y_train)

y_pred_bal = model_balanced.predict(X_test_scaled)
y_proba_bal = model_balanced.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred_bal))
print('ROC-AUC:', roc_auc_score(y_test, y_proba_bal))

              precision    recall  f1-score   support

           0       0.92      0.69      0.79     17605
           1       0.18      0.55      0.28      2263

    accuracy                           0.67     19868
   macro avg       0.55      0.62      0.53     19868
weighted avg       0.84      0.67      0.73     19868

ROC-AUC: 0.6687342142461185


## Gradient boosting comparison

No scaling needed — tree-based models are unaffected by feature scale, so we use the original unscaled X_train/X_test.

In [12]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb_model = HistGradientBoostingClassifier(class_weight='balanced', random_state=42)
gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)
y_proba_gb = gb_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_gb))
print('ROC-AUC:', roc_auc_score(y_test, y_proba_gb))

              precision    recall  f1-score   support

           0       0.92      0.69      0.79     17605
           1       0.19      0.57      0.28      2263

    accuracy                           0.67     19868
   macro avg       0.56      0.63      0.54     19868
weighted avg       0.84      0.67      0.73     19868

ROC-AUC: 0.6772852186797153


## Threshold tuning for gradient boosting

In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
for t in thresholds:
    y_pred_t = (y_proba_gb >= t).astype(int)
    precision = precision_score(y_test, y_pred_t)
    recall = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    print(f"threshold={t}: precision={precision:.2f}, recall={recall:.2f}, f1={f1:.2f}")

threshold=0.2: precision=0.12, recall=1.00, f1=0.21
threshold=0.3: precision=0.12, recall=0.96, f1=0.22
threshold=0.4: precision=0.15, recall=0.80, f1=0.25
threshold=0.5: precision=0.19, recall=0.57, f1=0.28
threshold=0.6: precision=0.24, recall=0.34, f1=0.28
threshold=0.7: precision=0.34, recall=0.14, f1=0.20


## Permutation importance for gradient boosting

In [14]:
from sklearn.inspection import permutation_importance

result = permutation_importance(gb_model, X_test, y_test, n_repeats=10, random_state=42, scoring='roc_auc')
importances = pd.Series(result.importances_mean, index=X_test.columns)
importances.sort_values(ascending=False).head(15)

number_inpatient                                0.079202
discharge_disposition_id_22                     0.019279
discharge_disposition_id_3                      0.018546
discharge_disposition_id_5                      0.007544
time_in_hospital                                0.004584
number_diagnoses                                0.004423
discharge_disposition_id_2                      0.003692
number_emergency                                0.003353
diabetesMed_Yes                                 0.002809
payer_code_Unknown                              0.002129
num_medications                                 0.001796
discharge_disposition_id_6                      0.001584
diag_1_cat_Other                                0.001568
medical_specialty_Orthopedics-Reconstructive    0.001379
medical_specialty_Emergency/Trauma              0.001024
dtype: float64

## Save the model

In [15]:
import joblib

joblib.dump(gb_model, '../models/readmission/gradient_boosting_v1.joblib')
joblib.dump(list(X_encoded.columns), '../models/readmission/feature_columns.joblib')

['../models/readmission/feature_columns.joblib']